## Conditional edges and loops

- A loop is only an edge pointing back at a node already visited
- What ends it is a conditional edge whose router decides to leave
- If nothing in the state moves forward, the router never changes its mind

The second half runs deliberately broken code so we see the error, then fixes
it.

### Exercise
Tool node: word_count
Modify the tool function below so that it counts letters instead of words.

In [ ]:
def word_count_tool(state: dict) -> dict:
    q = state.get("question","")
    state["wc"] = len(str(q).split())
    return state
assert word_count_tool({"question":"The cat sleeps"})["wc"] == 3

### Solution
The task is to change the tool from counting words to counting letters. The current code does len(str(q).split()) : split on spaces, count the pieces (words). To count letters, count the characters instead:

In [ ]:
def word_countletter_tool(state: dict) -> dict:
    q = state.get("question", "")
    state["wc"] = len(str(q).replace(" ", ""))   # count letters, not words
    return state

assert word_countletter_tool({"question": "The cat sleeps"})["wc"] == 12

### Exercise
Consider the following code which gives a GraphRecursionError. Can we explain why and solve the problem.

In [ ]:
# This cell is MEANT to fail. Read the error, then look at the next cell.
from langgraph.errors import GraphRecursionError

try:
    from typing import TypedDict, Optional
    from langgraph.graph import StateGraph, START, END

    class State(TypedDict, total=False):
        question: str
        wc: int
        answer: Optional[str]
        loops: int

    def word_count_tool(state: dict) -> dict:
        q = state.get("question", "")
        state["wc"] = len(str(q).split())
        return state

    def generator_node(state: State) -> State:
        state["answer"] = "Longer answer..." if state.get("wc", 0) > 4 else "Short answer."
        return state

    def need_more_detail(state: State) -> bool:
        return state.get("answer") == "Short answer."

    builder = StateGraph(State)
    builder.add_node("tool_wc", word_count_tool)
    builder.add_node("gen", generator_node)
    builder.add_edge(START, "tool_wc")
    builder.add_edge("tool_wc", "gen")

    def router(state: State) -> str:
        return "loopback" if need_more_detail(state) else END

    builder.add_conditional_edges("gen", router, {"loopback": "tool_wc", END: END})
    graph = builder.compile()

    # A short question loops forever → GraphRecursionError
    print(graph.invoke({"question": "Short one"}))
except GraphRecursionError as e:
    print("GraphRecursionError, exactly as expected:")
    print(str(e)[:200])

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END

class State(TypedDict, total=False):
    question: str
    wc: int
    answer: Optional[str]
    loops: int

def word_count_tool(state: dict) -> dict:
    q = state.get("question", "")
    state["wc"] = len(str(q).split())
    state["loops"] = state.get("loops", 0) + 1      # CHANGE 1: something advances each pass
    return state

def generator_node(state: State) -> State:
    state["answer"] = "Longer answer..." if state.get("wc", 0) > 4 else "Short answer."
    return state

def need_more_detail(state: State) -> bool:
    return state.get("answer") == "Short answer."

builder = StateGraph(State)
builder.add_node("tool_wc", word_count_tool)
builder.add_node("gen", generator_node)
builder.add_edge(START, "tool_wc")
builder.add_edge("tool_wc", "gen")

def router(state: State) -> str:
    if state.get("loops", 0) >= 3:                  # CHANGE 2: bounded exit the loop can reach
        return END
    return "loopback" if need_more_detail(state) else END

builder.add_conditional_edges("gen", router, {"loopback": "tool_wc", END: END})
graph = builder.compile()

print("Short question:")
print(graph.invoke({"question": "Short one"}))

print("\nLong question:")
print(graph.invoke({"question": "This is a much longer question with plenty of detail"}))

### Try breaking it again

- Set the counter increment back to 0 and re-run
- The recursion limit is what catches it. In an agent the same shape costs
  real money before it stops, which is why loops get a guard